# 10 — Can public data call the Brazilian curve's next month? (DUSTIN-BR Stage 4)

**The question:** *Notebook 09 built a point-in-time dataset for the Brazilian
rate curve and proved it never looks ahead. Does it predict where the curve
goes over the next 21 sessions?*

**The answer, first: not reliably.** Out of sample, from 2012 to 2026 (3,633
sessions), a regularised logistic model on 24 public features gives
probabilities no better than two naive baselines. That holds for the curve's
regime, for the 2y level alone and for the 2s5s slope alone:

| Target | Log loss: model / climatology / Markov | Hit rate: model / climatology / Markov |
| --- | --- | --- |
| Regime (4 classes) | 1.387 / 1.399 / 1.400 | 29.0% / 20.2% / 26.1% |
| Level (2y up or down) | 0.706 / 0.703 / 0.698 | 51.8% / 47.7% / 50.0% |
| Slope (2s5s wider or narrower) | 0.701 / 0.697 / 0.700 | 56.0% / 50.6% / 47.9% |

The one lead is the slope's direction, called right 56% of the time. That is
not distinguishably better than always calling the more common side. It
beats the persistence rule, but only because that rule is a coin for the
slope. With 12 comparisons, about one such result is expected by chance, so it
is something to track forward, not a finding.

This notebook walks through how that answer was reached, so you can check
it. Two kinds of cells:

* **Machinery cells** run anywhere, on synthetic data and the repository's
  own model code (`research_examples/dustin_br/model.py`).
* **Warehouse cells** need the real matrix: the `dustin_br.csv` that the
  **Research Build** workflow uploads (`.github/workflows/research_build.yml`;
  download the artifact from a run). Point `DUSTIN_BR_CSV` at it. Without it,
  those cells print the numbers recorded from run 36444747902 instead.

## Setup

In [1]:
# pip install -r notebooks/requirements.txt   (the model itself needs only numpy and pandas)
import os
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
while not (ROOT / "research_examples").is_dir() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from research_examples.dustin_br import model as m

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 30)

MATRIX = os.getenv("DUSTIN_BR_CSV")
frame = pd.read_csv(MATRIX, parse_dates=["date"]) if MATRIX and Path(MATRIX).is_file() else None
if frame is None:
    print("No matrix: warehouse cells will print the recorded results of run 36444747902.")
else:
    print(f"matrix: {len(frame):,} sessions, {frame['date'].min():%Y-%m-%d} to {frame['date'].max():%Y-%m-%d}")

# Chart tokens (dataviz reference palette; the three series slots validated for colour-vision deficiency).
SURFACE, INK, INK_2, MUTED, GRID, AXIS = "#fcfcfb", "#0b0b0b", "#52514e", "#898781", "#e1e0d9", "#c3c2b7"
SERIES = ["#2a78d6", "#eb6834", "#1baf7a"]


def style(ax, title):
    ax.figure.set_facecolor(SURFACE)
    ax.set_facecolor(SURFACE)
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
    for side in ("left", "bottom"):
        ax.spines[side].set_color(AXIS)
    ax.grid(True, color=GRID, lw=0.75)
    ax.set_axisbelow(True)
    ax.tick_params(colors=MUTED, labelcolor=INK_2)
    ax.set_title(title, color=INK, loc="left", fontsize=11)
    ax.xaxis.label.set_color(INK_2)
    ax.yaxis.label.set_color(INK_2)

No matrix: warehouse cells will print the recorded results of run 36444747902.


## Step 1 — What is predicted

Over the next 21 B3 sessions (about a month), the 2y DI rate falls (**bull**)
or rises (**bear**), and the 2s5s slope (5y minus 2y) widens (**steepener**)
or narrows (**flattener**). Three targets come from those two moves:

* **regime**: the four combinations (bull_steepener, bull_flattener,
  bear_steepener, bear_flattener);
* **level**: bull or bear;
* **slope**: steepener or flattener.

The label of session *t* is built from sessions *t* and *t + 21*. It looks
forward by definition, which is why it lives in the model code and never in
the dataset. A session whose rate did not move, or whose 21st session ahead
has not happened yet, has no label.

In [2]:
toy = pd.DataFrame({"di_2y": np.r_[np.linspace(10, 11, 30), np.linspace(11, 10.5, 30)],
                    "di_2s5s": np.r_[np.linspace(0.5, 0.2, 30), np.linspace(0.2, 0.6, 30)]})
labels = pd.DataFrame({name: target.label(toy).map(lambda c, t=target: t.classes[int(c)] if c == c else None)
                       for name, target in m.TARGETS.items()})
print(labels.iloc[[0, 10, 20, 29, 38, 39]].to_string())

if frame is not None:
    for name, target in m.TARGETS.items():
        share = target.label(frame).dropna().astype(int).map(dict(enumerate(target.classes)))
        print(f"\n{name}: " + ", ".join(f"{k} {v:.1%}" for k, v in share.value_counts(normalize=True).items()))

            regime level      slope
0   bear_flattener  bear  flattener
10  bear_flattener  bear  flattener
20  bear_steepener  bear  steepener
29  bull_steepener  bull  steepener
38  bull_steepener  bull  steepener
39            None  None       None


## Step 2 — Why 4,600 sessions are worth about 220

Two neighbouring labels share 20 of their 21 sessions, so they are almost the
same observation. The correlation between the 21-session changes *k* sessions
apart is about (21 − *k*) / 21 for any random walk, whatever the market does:
the overlap is mechanical. Only about 4,600 / 21 ≈ 220 of the outcomes are
independent. That number sets two things below. A model with 24 inputs
needs strong regularisation, and every uncertainty interval resamples whole
21-session blocks rather than single sessions.

In [3]:
rng = np.random.default_rng(0)
walk = pd.Series(np.cumsum(rng.normal(size=200_000)))
change = walk.shift(-21) - walk
lags = [1, 5, 10, 15, 20, 21, 30]
print(pd.DataFrame({"lag (sessions)": lags,
                    "correlation": [round(change.autocorr(k), 3) for k in lags],
                    "(21 - k) / 21": [round(max(21 - k, 0) / 21, 3) for k in lags]}).to_string(index=False))

 lag (sessions)  correlation  (21 - k) / 21
              1        0.952          0.952
              5        0.763          0.762
             10        0.528          0.524
             15        0.293          0.286
             20        0.057          0.048
             21        0.009          0.000
             30        0.006          0.000


## Step 3 — The inputs

The 24 inputs are CORE series from notebook 09, taken as spreads, changes,
volatilities and correlations. Raw rate levels are left out. SELIC ran from 2%
to 15% between 2008 and 2026, and a level the training window never saw
would be extrapolation. `policy_gap` is the 1y DI minus SELIC: how much policy
the curve prices over a year. `real_policy_rate` is SELIC minus 12-month IPCA.

In [4]:
groups = {
    "DI curve shape": ["di_2s5s", "di_curvature", "policy_gap", "real_policy_rate"],
    "DI momentum and volatility": [f for f in m.FEATURES if f.startswith(("di_2y_", "di_2s5s_"))],
    "Dollar (PTAX)": [f for f in m.FEATURES if f.startswith("usdbrl")],
    "U.S. Treasuries": [f for f in m.FEATURES if f.startswith("ust")] + ["rates_vol_proxy"],
    "Global stress (OFR FSI)": [f for f in m.FEATURES if f.startswith("ofr")],
    "Oil (Brent)": [f for f in m.FEATURES if f.startswith("brent")],
    "Cross-market correlation": [f for f in m.FEATURES if f.startswith("corr")],
}
assert sorted(sum(groups.values(), [])) == sorted(m.FEATURES)
for group, cols in groups.items():
    print(f"{group:28s} {', '.join(cols)}")

DI curve shape               di_2s5s, di_curvature, policy_gap, real_policy_rate
DI momentum and volatility   di_2y_mom_5d, di_2y_mom_21d, di_2y_mom_63d, di_2s5s_mom_21d, di_2s5s_mom_63d, di_2y_rv_21d, di_2y_rv_63d
Dollar (PTAX)                usdbrl_mom_21d, usdbrl_mom_63d, usdbrl_rv_21d
U.S. Treasuries              ust_2s10s, ust_2y_mom_21d, ust_10y_mom_21d, rates_vol_proxy
Global stress (OFR FSI)      ofr_fsi, ofr_fsi_mom_21d
Oil (Brent)                  brent_mom_21d, brent_mom_63d
Cross-market correlation     corr_di2y_usdbrl_63d, corr_di2y_ust10y_63d


## Step 4 — The model and the protocol

* **Model.** Multinomial logistic regression with an L2 penalty, fitted by
  Newton's method in numpy (`model.Logit`). It minimises the same objective as
  scikit-learn's `LogisticRegression`; one offline check agreed to within 1e-7
  in probability. Inputs are standardised on each training window alone.
* **Walk-forward.** For each test year from 2012, the model is fitted on every
  session whose label closes before the year starts. The 21 sessions just
  before the year are **purged**, because their labels reach into it. It then
  predicts every session of the year.
* **Two penalties, reported side by side.** C = 0.1 was set before any test
  year was scored. A second variant, added after the regime result, picks C on
  each training window's last two years and never sees a test year. It is the
  headline.
* **Two baselines on the same sessions.** *Climatology* is each class's share
  so far. *Markov* is each class's share after the class of the 21 sessions
  that just ended; it is the persistence rule.

The folds, on the real session calendar when the matrix is loaded (a weekday
calendar otherwise):

In [5]:
dates = (frame["date"] if frame is not None
         else pd.Series(pd.bdate_range("2008-01-02", "2026-09-25"))).reset_index(drop=True)
rows = []
for year in range(m.FIRST_TEST_YEAR, dates.dt.year.max() + 1):
    first = int(np.flatnonzero(dates.dt.year == year)[0])
    last_train = first - m.HORIZON - 1          # row i may train only if i + 21 < first
    rows.append({"test year": year, "last training session": f"{dates[last_train]:%Y-%m-%d}",
                 "purged": f"{dates[last_train + 1]:%Y-%m-%d} .. {dates[first - 1]:%Y-%m-%d}",
                 "first test session": f"{dates[first]:%Y-%m-%d}"})
print(pd.DataFrame(rows).to_string(index=False))

 test year last training session                   purged first test session
      2012            2011-12-01 2011-12-02 .. 2011-12-30         2012-01-02
      2013            2012-11-30 2012-12-03 .. 2012-12-31         2013-01-01
      2014            2013-12-02 2013-12-03 .. 2013-12-31         2014-01-01
      2015            2014-12-02 2014-12-03 .. 2014-12-31         2015-01-01
      2016            2015-12-02 2015-12-03 .. 2015-12-31         2016-01-01
      2017            2016-12-01 2016-12-02 .. 2016-12-30         2017-01-02
      2018            2017-11-30 2017-12-01 .. 2017-12-29         2018-01-01
      2019            2018-11-30 2018-12-03 .. 2018-12-31         2019-01-01
      2020            2019-12-02 2019-12-03 .. 2019-12-31         2020-01-01
      2021            2020-12-02 2020-12-03 .. 2020-12-31         2021-01-01
      2022            2021-12-02 2021-12-03 .. 2021-12-31         2022-01-03
      2023            2022-12-01 2022-12-02 .. 2022-12-30         2023-01-02

## Step 5 — Would the protocol find a signal if one were there?

A null result means little unless the method can find a real signal and does
not invent one. Here are three controls for the level target, on synthetic
sessions from 2008 to 2013, over six random seeds:

* **Noise.** Every input is stationary noise, and the labels come from a
  random walk. The model should gain nothing over climatology.
* **Planted signal.** One input carries the sign of the next 21 sessions'
  level change, plus noise. That leak is deliberate. The model should find it
  clearly.
* **Persistent inputs.** As *noise*, but two inputs are random walks, the way
  a raw rate level is. This is the hazard behind Step 3. A slow-moving input
  fits overlapping labels in sample by accident, and that fit fails out of
  sample.

The first two also run in CI (`tests/test_dustin_br_model.py`).

In [6]:
def synthetic(seed, planted=0.0, persistent=False, years=6):
    rng = np.random.default_rng(seed)
    dates = pd.bdate_range("2008-01-01", periods=250 * years)
    n = len(dates)
    di_2y = 10 + np.cumsum(rng.normal(0, 0.05, n))
    f = pd.DataFrame({"date": dates, "di_2y": di_2y, "di_2s5s": rng.normal(size=n),
                      "di_1y": 10.0 + rng.normal(0, 0.1, n), "selic": 10.0,
                      "ipca_12m": 4.0 + rng.normal(0, 0.1, n)})
    for col in m.FEATURES:
        if col not in f and col not in ("policy_gap", "real_policy_rate"):
            f[col] = rng.normal(size=n)
    if persistent:
        f["di_1y"] = 10.0 + np.cumsum(rng.normal(0, 0.05, n))      # a raw level: a random walk
        f["di_2s5s"] = np.cumsum(rng.normal(0, 0.02, n))           # a slow spread: another
    if planted:
        up = np.sign(pd.Series(di_2y).shift(-m.HORIZON) - pd.Series(di_2y)).fillna(0).to_numpy()
        f["ofr_fsi"] = planted * up + rng.normal(size=n)
    return f


level = m.TARGETS["level"]
rows = []
for seed in range(2, 8):
    for label, kw in (("noise", {}), ("planted signal", {"planted": 2.0}), ("persistent inputs", {"persistent": True})):
        preds = m.walk_forward(synthetic(seed, **kw), level)
        g, lo, hi = m.summary(preds, level.classes)["gains"][("log_loss", m.HEADLINE, "climatology")]
        rows.append({"seed": seed, "control": label, "gain over climatology [90%]": f"{g:+.3f} [{lo:+.3f}, {hi:+.3f}]"})
print(pd.DataFrame(rows).pivot(index="seed", columns="control",
                                values="gain over climatology [90%]")[["noise", "planted signal", "persistent inputs"]])

control                    noise           planted signal        persistent inputs
seed                                                                              
2        -0.003 [-0.009, +0.002]  +0.540 [+0.514, +0.570]  -0.128 [-0.192, -0.062]
3        +0.000 [-0.002, +0.003]  +0.592 [+0.563, +0.617]  -0.002 [-0.004, +0.000]
4        -0.000 [-0.001, +0.000]  +0.598 [+0.580, +0.616]  +0.005 [-0.006, +0.014]
5        -0.000 [-0.002, +0.001]  +0.581 [+0.556, +0.607]  +0.000 [-0.001, +0.001]
6        -0.003 [-0.007, -0.000]  +0.607 [+0.570, +0.640]  -0.001 [-0.002, -0.000]
7        -0.001 [-0.004, +0.002]  +0.614 [+0.566, +0.680]  -0.060 [-0.115, -0.023]


Noise earns about zero on every seed, and the planted signal is found on
every seed. Persistent inputs lose on some seeds and never gain distinguishably,
which is why
the real inputs are spreads, changes and volatilities.

## Step 6 — The results on the warehouse

Scores, lower is better for both loss measures:

* **Log loss** measures how good the probabilities are. A coin scores 1.386
  on four classes and 0.693 on two.
* **Brier score** measures the same thing on a squared scale.
* **Hit rate** is how often the most likely class happened.

A *gain* is the baseline's log loss minus the model's, or the model's hit rate
minus the baseline's, so above 0 means the model is better. Its 90% interval
comes from resampling 21-session blocks (Step 2). This cell refits 15 years
for each of three targets and takes a couple of minutes.

In [7]:
RECORDED = pd.DataFrame(  # run 36444747902, model with C chosen in training
    [("regime", 1.387, 1.399, 1.400, "+0.013 [-0.014, +0.037]", 0.290, 0.202, 0.261, "+2.9 [-1.4, +7.0]"),
     ("level", 0.706, 0.703, 0.698, "-0.008 [-0.029, +0.011]", 0.518, 0.477, 0.500, "+1.8 [-3.3, +6.6]"),
     ("slope", 0.701, 0.697, 0.700, "-0.001 [-0.031, +0.026]", 0.560, 0.506, 0.479, "+8.0 [+1.9, +14.3]")],
    columns=["target", "log loss: model", "climatology", "Markov", "log-loss gain vs Markov",
             "hit: model", "climatology ", "Markov ", "hit gain vs Markov, pp"])

results = {}
if frame is None:
    print("Recorded (run 36444747902):")
    print(RECORDED.to_string(index=False))
else:
    rows = []
    for name, target in m.TARGETS.items():
        preds = m.walk_forward(frame, target)
        s = m.summary(preds, target.classes)
        results[name] = (preds, s)
        r = s["table"].iloc[-1]
        g, lo, hi = s["gains"][("log_loss", m.HEADLINE, "Markov")]
        h, hlo, hhi = s["gains"][("hit", m.HEADLINE, "Markov")]
        rows.append((name, *(round(r[f"log_loss:{n}"], 3) for n in (m.HEADLINE, "climatology", "Markov")),
                     f"{g:+.3f} [{lo:+.3f}, {hi:+.3f}]",
                     *(round(r[f"hit:{n}"], 3) for n in (m.HEADLINE, "climatology", "Markov")),
                     f"{100 * h:+.1f} [{100 * hlo:+.1f}, {100 * hhi:+.1f}]"))
    print(pd.DataFrame(rows, columns=RECORDED.columns).to_string(index=False))

Recorded (run 36444747902):
target  log loss: model  climatology  Markov log-loss gain vs Markov  hit: model  climatology   Markov  hit gain vs Markov, pp
regime            1.387        1.399   1.400 +0.013 [-0.014, +0.037]       0.290         0.202    0.261      +2.9 [-1.4, +7.0]
 level            0.706        0.703   0.698 -0.008 [-0.029, +0.011]       0.518         0.477    0.500      +1.8 [-3.3, +6.6]
 slope            0.701        0.697   0.700 -0.001 [-0.031, +0.026]       0.560         0.506    0.479     +8.0 [+1.9, +14.3]


By era, for the model with C chosen in training, climatology and Markov:

In [8]:
if results:
    for name, (preds, s) in results.items():
        t = s["table"].set_index("era")
        print(f"\n{name}")
        print(pd.DataFrame({
            "rows": t["rows"],
            "log loss (model / clim / Markov)": [" / ".join(f"{r[f'log_loss:{n}']:.3f}" for n in (m.HEADLINE, "climatology", "Markov")) for _, r in t.iterrows()],
            "hit (model / clim / Markov)": [" / ".join(f"{r[f'hit:{n}']:.1%}" for n in (m.HEADLINE, "climatology", "Markov")) for _, r in t.iterrows()],
        }).to_string())
else:
    print("Needs the matrix (DUSTIN_BR_CSV). Slope, recorded: 2012-2016 56.0%, 2017-2020 46.6%, "
          "2021-2026 62.5% hit rate against 54.9% / 44.2% / 51.3% for climatology.")

Needs the matrix (DUSTIN_BR_CSV). Slope, recorded: 2012-2016 56.0%, 2017-2020 46.6%, 2021-2026 62.5% hit rate against 54.9% / 44.2% / 51.3% for climatology.


## Step 7 — Calibration: are the probabilities honest?

A probability is useful when "70%" happens about 70% of the time. Each chart
pools every class and puts each prediction in a bin. It plots the mean
predicted probability against how often that class happened. The diagonal is
perfect calibration. C = 0.1 is badly overconfident. C chosen in training is
close to the diagonal near the middle and still overconfident at the ends.

In [9]:
if results:
    fig, axes = plt.subplots(1, 3, figsize=(13, 4.2), sharey=True)
    for ax, (name, (preds, _)) in zip(axes, results.items()):
        classes = m.TARGETS[name].classes
        ax.plot([0, 1], [0, 1], color=MUTED, lw=0.75)
        ax.annotate("perfectly calibrated", (0.64, 0.72), rotation=38, color=MUTED, fontsize=8)
        for color, (label, prefix), short, dy in zip(SERIES, m.MODELS.items(), ("C = 0.1", "chosen C"), (-10, 6)):
            cal = m.calibration(preds, prefix, classes)
            ax.plot(cal["predicted"], cal["happened"], color=color, lw=1.5, marker="o", ms=6,
                    mec=SURFACE, mew=1.5, label=label.replace("model, ", ""), solid_capstyle="round")
            ax.annotate(short, (cal["predicted"].iloc[-1], cal["happened"].iloc[-1]),
                        xytext=(7, dy), textcoords="offset points", ha="left", color=INK_2, fontsize=8)
        ax.set(xlim=(0, 1), ylim=(0, 1), xlabel="predicted probability")
        style(ax, name)
    axes[0].set_ylabel("share that happened")
    axes[0].legend(frameon=False, labelcolor=INK_2, loc="upper left")
    fig.tight_layout()
    plt.show()
else:
    print("Needs the matrix (DUSTIN_BR_CSV). Slope, recorded: 74% predicted, 60% happened (C chosen in training).")

Needs the matrix (DUSTIN_BR_CSV). Slope, recorded: 74% predicted, 60% happened (C chosen in training).


## Step 8 — Where the gains come from, over time

The running sum of the log-loss gain over the Markov baseline, one session at
a time. A real edge climbs steadily. These lines wander around zero, then
drift: each target's gains in one stretch are given back in another.

In [10]:
if results:
    fig, ax = plt.subplots(figsize=(11, 4.2))
    ax.axhline(0, color=AXIS, lw=0.75)
    for color, (name, (preds, _)) in zip(SERIES, results.items()):
        classes = m.TARGETS[name].classes
        model_rows = m.row_scores(preds, "n_", classes)
        markov_rows = m.row_scores(preds, "markov_", classes)
        cum = (markov_rows["log_loss"] - model_rows["log_loss"]).cumsum()
        dates = pd.to_datetime(model_rows["date"])
        ax.plot(dates, cum, color=color, lw=1.5, label=name, solid_capstyle="round")
        ax.annotate(name, (dates.iloc[-1], cum.iloc[-1]), xytext=(6, 0), textcoords="offset points",
                    va="center", color=INK_2, fontsize=9)
    ax.set_ylabel("cumulative log-loss gain over Markov")
    style(ax, "Cumulative gain of the model (C chosen in training) over the Markov baseline")
    ax.legend(frameon=False, labelcolor=INK_2, loc="upper left")
    fig.tight_layout()
    plt.show()
else:
    print("Needs the matrix (DUSTIN_BR_CSV).")

Needs the matrix (DUSTIN_BR_CSV).


## Step 9 — Reading the slope lead

The slope's direction is the one result that clears an interval, and it is
weaker than it looks:

* **It clears the interval only against Markov, and Markov is a coin here.** The
  slope's last 21 sessions say nothing about its next 21 (the table below:
  about 51% flatten after either move). The persistence rule's out-of-sample
  hit rate, 47.9%, is a coin that landed low, so beating it by 8.0 points
  proves little. Against always calling the more common side, the gain is
  +5.4 points, with a 90% interval of -0.4 to +10.6: not distinguishable.
* **Its probabilities are no better.** Its log loss is not distinguishable from
  either baseline, because it is overconfident when it leans (Step 7).
* **Twelve comparisons.** There are 3 targets, 2 baselines and 2 metrics for
  the headline model, each at a one-sided 5% level. About one "better" is
  expected by chance alone.

The slope's last 21 sessions against its next 21, over every labelled
session:

In [11]:
if frame is not None:
    slope = m.TARGETS["slope"]
    both = pd.DataFrame({"before": slope.past(frame), "after": slope.label(frame)}).dropna().astype(int)
    table = pd.crosstab(both["before"].map(dict(enumerate(slope.classes))),
                        both["after"].map(dict(enumerate(slope.classes))), normalize="index")
    print("share of the next 21 sessions' slope move, by the last 21 sessions' move:")
    print(table.round(3).to_string())
else:
    print("Needs the matrix (DUSTIN_BR_CSV).")

Needs the matrix (DUSTIN_BR_CSV).


## Step 10 — Today's call, and how much to trust it

The headline model refitted on every label known at the last session. Read
it against Step 6: these are research outputs close to the base rates, not
signals.

In [12]:
if frame is not None:
    for name, target in m.TARGETS.items():
        day, c, p = m.latest_forecast(frame, target)
        print(f"{day} {name:7s} (C = {c:g}): " + ", ".join(f"{k} {v:.1%}" for k, v in zip(target.classes, p)))
else:
    print("Recorded, 2026-09-25: regime 26.6 / 23.4 / 22.7 / 27.3% (bull_steepener, bull_flattener, "
          "bear_steepener, bear_flattener); level bull 48.9%; slope steepener 49.7%.")

Recorded, 2026-09-25: regime 26.6 / 23.4 / 22.7 / 27.3% (bull_steepener, bull_flattener, bear_steepener, bear_flattener); level bull 48.9%; slope steepener 49.7%.


## What this means

* **The public inputs do not forecast the curve a month out** in a way that
  beats knowing the base rates. That is consistent with a liquid market that
  prices public information quickly. A linear model on these inputs is not
  enough to say otherwise.
* **Overfitting is the first danger.** With about 220 independent outcomes,
  C = 0.1, set in advance, made every target worse than the baselines. The
  penalty chosen inside each training window was 1e-4 to 1e-2: almost all
  shrinkage.
* **What could change the answer**, in order of cost:
  1. **Track it forward.** Research Build refits and records a forecast on
     every run. The slope lead either holds on sessions no one has seen, or
     it does not.
  2. **A longer history.** B3's reference files go back to 2004 (open in the
     research doc), which would add about 1,000 sessions.
  3. **Other inputs.** Focus survey vintages (SILO keeps none), and licensed
     series (VIX, MOVE, CDS), which the warehouse does not carry.
  4. **Non-linear models**, only with the same protocol and a reason to expect
     an interaction. With about 220 outcomes they overfit more easily than
     logistic regression.

## What this notebook did not do

* **It did not tune on the test years.** C = 0.1 was fixed first. The second
  variant was added after its failure and never sees a test year. The level
  and slope targets were fixed before they were scored. Every variant is
  reported.
* **It gives no trading advice.** A 56% hit rate on overlapping monthly calls
  is not a strategy, and costs are not modelled.

Reproduce it: dispatch **Research Build** (it runs `quality.py`, then
`model.py`), download the artifact, and run this notebook with `DUSTIN_BR_CSV`
pointing at its `dustin_br.csv`.